# Stock Price Data Exploration

This notebook explores the stock price data and sentiment data for Amazon (AMZN) stock forecasting.

## Objectives:
1. Load and examine stock price data
2. Analyze sentiment data from Twitter
3. Explore correlations between sentiment and stock movements
4. Visualize key patterns and trends

In [ ]:
# Import required libraries
import sys
import os

# Add src directory to path
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Import our custom modules
from data_pipeline.stock_fetch import StockDataFetcher
from data_pipeline.twitter_fetch import TwitterDataFetcher
from sentiment.sentiment import SentimentAnalyzer
from visualization.visualization import StockVisualization

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

## 1. Load Stock Data

In [ ]:
# Initialize stock data fetcher for Amazon
stock_fetcher = StockDataFetcher('AMZN')

# Fetch 6 months of hourly data
print("Fetching AMZN stock data...")
stock_data = stock_fetcher.fetch_historical_data(period='6mo', interval='1h')

print(f"Fetched {len(stock_data)} records")
print(f"Date range: {stock_data['date'].min()} to {stock_data['date'].max()}")
stock_data.head()

In [ ]:
# Basic statistics
print("Stock Data Summary:")
print(stock_data.describe())

# Check for missing values
print("\nMissing values:")
print(stock_data.isnull().sum())

In [ ]:
# Calculate returns and volatility
stock_data = stock_fetcher.calculate_returns(stock_data)

print(f"Average daily return: {stock_data['daily_return'].mean():.4f}")
print(f"Daily volatility: {stock_data['daily_return'].std():.4f}")
print(f"Annualized volatility: {stock_data['daily_return'].std() * np.sqrt(252):.4f}")

## 2. Visualize Stock Data

In [ ]:
# Initialize visualizer
viz = StockVisualization(figsize=(15, 10))

# Plot comprehensive stock analysis
viz.plot_stock_data(
    stock_data,
    price_cols=['open', 'high', 'low', 'close'],
    title="Amazon (AMZN) Stock Analysis - 6 Months"
)

## 3. Sentiment Data Analysis

In [ ]:
# Note: For this demo, we'll create mock sentiment data
# In production, you would use: 
# twitter_fetcher = TwitterDataFetcher("YOUR_BEARER_TOKEN")
# tweets = twitter_fetcher.fetch_tweets()

# Create mock sentiment data that correlates with stock movements
np.random.seed(42)

# Generate hourly sentiment data
start_date = stock_data['date'].min()
end_date = stock_data['date'].max()
hourly_dates = pd.date_range(start=start_date, end=end_date, freq='1H')

# Create sentiment that has some correlation with price movements
price_changes = stock_data.set_index('date')['close'].pct_change().reindex(hourly_dates, method='ffill')
base_sentiment = np.random.normal(0, 0.2, len(hourly_dates))
correlated_sentiment = base_sentiment + 0.3 * price_changes.fillna(0)

mock_sentiment_data = pd.DataFrame({
    'created_at': hourly_dates,
    'sentiment_index': correlated_sentiment,
    'sentiment_mean': correlated_sentiment + np.random.normal(0, 0.1, len(hourly_dates)),
    'tweet_count': np.random.randint(5, 100, len(hourly_dates)),
    'sentiment_momentum': np.gradient(correlated_sentiment)
})

print(f"Generated {len(mock_sentiment_data)} sentiment records")
mock_sentiment_data.head()

In [ ]:
# Sentiment analysis statistics
print("Sentiment Data Summary:")
print(mock_sentiment_data.describe())

# Plot sentiment analysis
viz.plot_sentiment_analysis(
    mock_sentiment_data,
    stock_data,
    title="Amazon Stock Sentiment Analysis"
)

## 4. Correlation Analysis

In [ ]:
# Merge sentiment data with stock data for correlation analysis
# Round timestamps to nearest hour for joining
stock_hourly = stock_data.copy()
stock_hourly['hour'] = stock_hourly['date'].dt.floor('H')
sentiment_hourly = mock_sentiment_data.copy()
sentiment_hourly['hour'] = sentiment_hourly['created_at'].dt.floor('H')

# Aggregate stock data by hour (take last value)
stock_agg = stock_hourly.groupby('hour').last().reset_index()

# Merge datasets
merged_data = pd.merge(stock_agg, sentiment_hourly, on='hour', how='inner')

print(f"Merged dataset shape: {merged_data.shape}")
print(f"Date range: {merged_data['hour'].min()} to {merged_data['hour'].max()}")

In [ ]:
# Calculate correlations between sentiment and stock metrics
correlation_cols = [
    'close', 'volume', 'daily_return', 'volatility_20d',
    'sentiment_index', 'sentiment_mean', 'tweet_count', 'sentiment_momentum'
]

correlation_data = merged_data[correlation_cols].dropna()
correlation_matrix = correlation_data.corr()

# Display key correlations
sentiment_correlations = correlation_matrix['sentiment_index'].drop('sentiment_index')
print("Correlations with Sentiment Index:")
print(sentiment_correlations.sort_values(key=abs, ascending=False))

In [ ]:
# Plot correlation matrix
viz.plot_correlation_matrix(
    correlation_data,
    title="Stock Price vs Sentiment Correlation Matrix"
)

## 5. Time Series Analysis

In [ ]:
# Analyze daily patterns
merged_data['hour_of_day'] = merged_data['hour'].dt.hour
merged_data['day_of_week'] = merged_data['hour'].dt.dayofweek

# Average returns by hour of day
hourly_returns = merged_data.groupby('hour_of_day')['daily_return'].mean()
hourly_sentiment = merged_data.groupby('hour_of_day')['sentiment_index'].mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Plot hourly patterns
axes[0].plot(hourly_returns.index, hourly_returns.values, marker='o')
axes[0].set_title('Average Returns by Hour of Day')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Return')
axes[0].grid(True, alpha=0.3)

axes[1].plot(hourly_sentiment.index, hourly_sentiment.values, marker='o', color='red')
axes[1].set_title('Average Sentiment by Hour of Day')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Average Sentiment')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Weekly patterns
daily_returns = merged_data.groupby('day_of_week')['daily_return'].mean()
daily_sentiment = merged_data.groupby('day_of_week')['sentiment_index'].mean()

days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

axes[0].bar(days, daily_returns.values, alpha=0.7)
axes[0].set_title('Average Returns by Day of Week')
axes[0].set_ylabel('Average Return')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(True, alpha=0.3)

axes[1].bar(days, daily_sentiment.values, alpha=0.7, color='red')
axes[1].set_title('Average Sentiment by Day of Week')
axes[1].set_ylabel('Average Sentiment')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Key Findings Summary

In [ ]:
# Summary statistics
print("=== KEY FINDINGS SUMMARY ===")
print(f"\n📊 Dataset Overview:")
print(f"  • Stock records: {len(stock_data):,}")
print(f"  • Sentiment records: {len(mock_sentiment_data):,}")
print(f"  • Date range: {stock_data['date'].min().strftime('%Y-%m-%d')} to {stock_data['date'].max().strftime('%Y-%m-%d')}")

print(f"\n💹 Stock Metrics:")
print(f"  • Price range: ${stock_data['close'].min():.2f} - ${stock_data['close'].max():.2f}")
print(f"  • Average daily return: {stock_data['daily_return'].mean()*100:.3f}%")
print(f"  • Daily volatility: {stock_data['daily_return'].std()*100:.3f}%")
print(f"  • Annualized volatility: {stock_data['daily_return'].std() * np.sqrt(252)*100:.1f}%")

print(f"\n💭 Sentiment Metrics:")
print(f"  • Average sentiment: {mock_sentiment_data['sentiment_index'].mean():.3f}")
print(f"  • Sentiment volatility: {mock_sentiment_data['sentiment_index'].std():.3f}")
print(f"  • Average tweets per hour: {mock_sentiment_data['tweet_count'].mean():.0f}")

print(f"\n🔗 Key Correlations:")
price_sentiment_corr = np.corrcoef(merged_data['daily_return'].dropna(), 
                                  merged_data['sentiment_index'].dropna())[0,1]
volume_sentiment_corr = np.corrcoef(merged_data['volume'].dropna(), 
                                   merged_data['tweet_count'].dropna())[0,1]
print(f"  • Price returns vs Sentiment: {price_sentiment_corr:.3f}")
print(f"  • Volume vs Tweet count: {volume_sentiment_corr:.3f}")

print(f"\n⚡ Next Steps:")
print(f"  • Add technical indicators for feature engineering")
print(f"  • Prepare sequences for GAN training")
print(f"  • Implement and train forecasting model")
print(f"  • Evaluate model performance")

## Notes for Next Notebooks:

1. **Feature Engineering** (02_feature_engineering.ipynb):
   - Add technical indicators (SMA, EMA, RSI, MACD, etc.)
   - Create lag features and rolling statistics
   - Normalize and scale features for model training

2. **Model Training** (03_model_training.ipynb):
   - Prepare sequences for GAN training
   - Train Generator and Discriminator
   - Monitor training progress and adjust hyperparameters

3. **Model Evaluation** (04_model_evaluation.ipynb):
   - Evaluate model predictions
   - Compare with baseline models
   - Analyze prediction accuracy and trading performance

**Data Quality Notes:**
- No missing values in stock data ✅
- Sentiment data is currently synthetic (replace with real Twitter data in production)
- Consider market hours for more realistic analysis
- Weekend data may need special handling